In [ ]:
import os
import shutil

import numpy as np
import pandas as pd
import xarray as xr
from cdo import Cdo
from joblib import Parallel, delayed


# Make the Levante CDO available to the python-cdo wrapper.
os.environ["PATH"] = (
    "/path/to/cdo/bin"
    + os.pathsep
    + os.environ.get("PATH", "")
)


# ---------------------------------------------------------------------------
# Event setup
# ---------------------------------------------------------------------------
# Change these two dates for a different recent event.
START_DATE = "2026-05-15"
END_DATE = "2026-06-01"

REGION_NAME = "western_europe"
EVENT_NAME = f"era5t_{START_DATE}_to_{END_DATE}"

# Western Europe:
# Iberia/UK/Ireland/France/Benelux/Germany/Alps/N. Italy/S. Scandinavia.
LON_MIN, LON_MAX = -40, 30
LAT_MIN, LAT_MAX = 20, 80

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
# ERA5T locations can vary by system. These defaults mirror the ERA5 layout
# used in the long historical workflow, with ERA5 replaced by ERA5T.
ERA5T_ROOT = "/pool/data/ERA5/ET"

HOUR_PATH = {
    "sf": f"{ERA5T_ROOT}/sf/an/1H/",
    "pl": f"{ERA5T_ROOT}/pl/an/1H/",
    "ml": f"{ERA5T_ROOT}/ml/an/1H/",
}

SCRATCH_PATH = f"${SCRATCH}/{REGION_NAME}/era5t_events/{EVENT_NAME}/"
FINAL_PATH = (
    f"${WORK}/02_MSE/{REGION_NAME}/"
    f"era5t_events/{EVENT_NAME}/raw/"
)
COMPILED_FILE = (
    f"${WORK}/02_MSE/{REGION_NAME}/"
    f"era5t_events/{EVENT_NAME}/era5t_{REGION_NAME}_{START_DATE}_to_{END_DATE}"
    "_compiled.nc"
)

N_JOBS_VARIABLES = 50


# First compute daily tasmax and tasmax_hour from 2m temperature.
TASMAX_CONFIG = {
    "var_num": "167",
    "var": "2t",
    "levels": "sf",
    "level": None,
}


# Then compute:
#   q: daily minimum and q at tasmax
#   t: daily maximum and value at tasmax
#   z: daily maximum, daily mean, and value at tasmax
VARIABLE_CONFIGS = [
    {
        "var_num": "133",
        "var": "q",
        "levels": "ml",
        "level": "137",
        "aggregations": ("daily_min", "at_tasmax"),
    },
    {
        "var_num": "130",
        "var": "t",
        "levels": "pl",
        "level": "50000",
        "aggregations": ("daily_max", "at_tasmax"),
    },
    {
        "var_num": "129",
        "var": "z",
        "levels": "pl",
        "level": "50000",
        "aggregations": ("daily_max", "daily_mean", "at_tasmax"),
    },
]


def event_days():
    return pd.date_range(start=START_DATE, end=END_DATE, freq="D")


def safe_remove(path):
    if path and os.path.exists(path):
        os.remove(path)


def ensure_output_dirs(var_name):
    var_scratch = os.path.join(SCRATCH_PATH, var_name)
    var_tmp = os.path.join(var_scratch, "temp")
    var_final = os.path.join(FINAL_PATH, var_name)
    os.makedirs(var_tmp, exist_ok=True)
    os.makedirs(var_final, exist_ok=True)
    return var_tmp, var_final


def tasmax_output_path():
    return os.path.join(
        FINAL_PATH,
        "tasmax",
        f"tasmax_{EVENT_NAME}_{REGION_NAME}.nc",
    )


def variable_output_path(var_name, aggregation):
    suffix = {
        "daily_min": "dailymin",
        "daily_max": "dailymax",
        "daily_mean": "dailymean",
        "at_tasmax": "at_tasmax",
    }[aggregation]
    return os.path.join(
        FINAL_PATH,
        var_name,
        f"{var_name}_{EVENT_NAME}_{suffix}_{REGION_NAME}.nc",
    )


def output_variable_name(var_name, aggregation):
    suffix = {
        "daily_min": "min",
        "daily_max": "max",
        "daily_mean": "mean",
        "at_tasmax": "at_tasmax",
    }[aggregation]
    return f"{var_name}_{suffix}"


def hourly_grib_path(config, day):
    date_str = day.strftime("%Y-%m-%d")
    return os.path.join(
        HOUR_PATH[config["levels"]],
        config["var_num"],
        f"ET{config['levels']}00_1H_{date_str}_{config['var_num']}.grb",
    )


def select_pressure_level(ds, level_value):
    """Select pressure levels whether CDO exposes them in Pa or hPa."""
    if level_value is None:
        return ds

    level_pa = int(level_value)
    candidates = [
        ("plev", level_pa),
        ("isobaricInhPa", level_pa // 100 if level_pa > 2000 else level_pa),
        ("level", level_pa),
    ]

    for coord_name, value in candidates:
        if coord_name in ds.coords:
            return ds.sel({coord_name: value})

    return ds


def squeeze_singleton_vertical_dims(ds):
    horizontal_and_time = {"time", "lat", "latitude", "lon", "longitude"}
    singleton_dims = [
        dim
        for dim, size in ds.sizes.items()
        if dim not in horizontal_and_time and size == 1
    ]
    if singleton_dims:
        return ds.squeeze(singleton_dims, drop=True)
    return ds


def convert_hourly_day_to_box_file(cdo, config, day, tmp_dir, product_name):
    """Convert one ERA5T hourly GRIB day to a regular-grid NetCDF region box."""
    date_str = day.strftime("%Y-%m-%d")
    grib_file = hourly_grib_path(config, day)

    if not os.path.exists(grib_file):
        print(f"Skipping missing: {grib_file}")
        return None

    stem = f"{product_name}_{config['var']}_{date_str}"
    reg_file = os.path.join(tmp_dir, f"reg_{stem}.nc")
    box_file = os.path.join(tmp_dir, f"box_{stem}.nc")
    level_file = os.path.join(tmp_dir, f"level_{stem}.nc")

    cdo.setgridtype(
        "regular",
        input=grib_file,
        output=reg_file,
        options="-O -f nc --eccodes",
    )

    cdo.sellonlatbox(
        LON_MIN,
        LON_MAX,
        LAT_MIN,
        LAT_MAX,
        input=reg_file,
        output=box_file,
        options="-O",
    )
    safe_remove(reg_file)

    if config["levels"] == "ml":
        cdo.sellevel(
            config["level"],
            input=box_file,
            output=level_file,
            options="-O",
        )
        safe_remove(box_file)
    elif config["levels"] == "pl":
        with xr.open_dataset(box_file) as ds:
            ds_selected = select_pressure_level(ds, config["level"])
            ds_selected = squeeze_singleton_vertical_dims(ds_selected)
            ds_selected = ds_selected.load()
            ds_selected["time"] = pd.to_datetime(ds_selected["time"].values)
            ds_selected.to_netcdf(level_file)
        safe_remove(box_file)
    else:
        safe_remove(level_file)
        shutil.move(box_file, level_file)

    return level_file


def to_daily_field(data_array, day, output_name):
    daily = data_array.rename(output_name)
    if "time" in daily.coords:
        daily = daily.reset_coords("time", drop=True)
    return daily.expand_dims(time=[pd.Timestamp(day.strftime("%Y-%m-%d"))])


def daily_extreme_dataset(x, day, mode, value_name, hour_name):
    if mode == "max":
        index = x.argmax(dim="time")
    elif mode == "min":
        index = x.argmin(dim="time")
    else:
        raise ValueError("mode must be 'max' or 'min'")

    value = x.isel(time=index)
    hour = x["time"].isel(time=index).dt.hour

    value = to_daily_field(value, day, value_name)
    hour = to_daily_field(hour, day, hour_name)

    return xr.Dataset({value_name: value, hour_name: hour}).load()


def daily_mean_dataset(x, day, value_name):
    """Daily mean over all hourly timesteps of the day."""
    value = x.mean(dim="time", keep_attrs=True)
    value = to_daily_field(value, day, value_name)
    return xr.Dataset({value_name: value}).load()


def sample_at_tasmax_dataset(x, day, tasmax_hour, output_name):
    day_start = np.datetime64(day.strftime("%Y-%m-%dT00:00:00"), "ns")
    tasmax_hour_int = tasmax_hour.round().astype("int16")

    target_time = xr.apply_ufunc(
        lambda hour: day_start + np.timedelta64(int(hour), "h"),
        tasmax_hour_int,
        vectorize=True,
        output_dtypes=[np.dtype("datetime64[ns]")],
    )

    sampled = x.sel(time=target_time, method="nearest")
    sampled = to_daily_field(sampled, day, output_name)
    return xr.Dataset({output_name: sampled}).load()


def process_tasmax_event():
    tmp_dir, _ = ensure_output_dirs("tasmax")
    output_file = tasmax_output_path()
    cdo = Cdo()

    daily_datasets = []
    for day in event_days():
        day_file = convert_hourly_day_to_box_file(
            cdo=cdo,
            config=TASMAX_CONFIG,
            day=day,
            tmp_dir=tmp_dir,
            product_name="tasmax",
        )
        if day_file is None:
            continue

        with xr.open_dataset(day_file) as ds:
            ds = squeeze_singleton_vertical_dims(ds)
            x = ds[TASMAX_CONFIG["var"]]
            daily = daily_extreme_dataset(
                x=x,
                day=day,
                mode="max",
                value_name="tasmax",
                hour_name="tasmax_hour",
            )
            daily_datasets.append(daily)

        safe_remove(day_file)

    if not daily_datasets:
        raise RuntimeError(f"No valid ERA5T tasmax files for {START_DATE} to {END_DATE}")

    ds_event = xr.concat(daily_datasets, dim="time")
    ds_event.attrs["source"] = "ERA5T"
    ds_event.attrs["event"] = EVENT_NAME
    ds_event.attrs["region"] = REGION_NAME
    ds_event.attrs["lon_bounds"] = f"{LON_MIN}, {LON_MAX}"
    ds_event.attrs["lat_bounds"] = f"{LAT_MIN}, {LAT_MAX}"
    ds_event.to_netcdf(output_file)
    ds_event.close()
    return output_file


def process_variable_event(config):
    var_name = config["var"]
    tmp_dir, _ = ensure_output_dirs(var_name)
    cdo = Cdo()

    tasmax_file = tasmax_output_path()
    if not os.path.exists(tasmax_file):
        raise FileNotFoundError(
            f"Missing tasmax file needed for at_tasmax sampling: {tasmax_file}"
        )

    daily_datasets = {aggregation: [] for aggregation in config["aggregations"]}

    with xr.open_dataset(tasmax_file) as tasmax_ds:
        for day in event_days():
            day_file = convert_hourly_day_to_box_file(
                cdo=cdo,
                config=config,
                day=day,
                tmp_dir=tmp_dir,
                product_name="var",
            )
            if day_file is None:
                continue

            with xr.open_dataset(day_file) as ds:
                ds = squeeze_singleton_vertical_dims(ds)
                x = ds[var_name]

                if "daily_min" in config["aggregations"]:
                    value_name = output_variable_name(var_name, "daily_min")
                    daily_datasets["daily_min"].append(
                        daily_extreme_dataset(
                            x=x,
                            day=day,
                            mode="min",
                            value_name=value_name,
                            hour_name=f"{value_name}_hour",
                        )
                    )

                if "daily_max" in config["aggregations"]:
                    value_name = output_variable_name(var_name, "daily_max")
                    daily_datasets["daily_max"].append(
                        daily_extreme_dataset(
                            x=x,
                            day=day,
                            mode="max",
                            value_name=value_name,
                            hour_name=f"{value_name}_hour",
                        )
                    )

                if "daily_mean" in config["aggregations"]:
                    value_name = output_variable_name(var_name, "daily_mean")
                    daily_datasets["daily_mean"].append(
                        daily_mean_dataset(
                            x=x,
                            day=day,
                            value_name=value_name,
                        )
                    )

                if "at_tasmax" in config["aggregations"]:
                    value_name = output_variable_name(var_name, "at_tasmax")
                    tasmax_hour = tasmax_ds["tasmax_hour"].sel(time=day)
                    daily_datasets["at_tasmax"].append(
                        sample_at_tasmax_dataset(
                            x=x,
                            day=day,
                            tasmax_hour=tasmax_hour,
                            output_name=value_name,
                        )
                    )

            safe_remove(day_file)

    output_files = {}
    for aggregation, datasets in daily_datasets.items():
        if not datasets:
            print(f"No valid ERA5T {var_name} files for {aggregation}")
            output_files[aggregation] = None
            continue

        ds_event = xr.concat(datasets, dim="time")
        ds_event.attrs["source"] = "ERA5T"
        ds_event.attrs["event"] = EVENT_NAME
        ds_event.attrs["region"] = REGION_NAME
        ds_event.attrs["lon_bounds"] = f"{LON_MIN}, {LON_MAX}"
        ds_event.attrs["lat_bounds"] = f"{LAT_MIN}, {LAT_MAX}"

        out_file = variable_output_path(var_name, aggregation)
        ds_event.to_netcdf(out_file)
        ds_event.close()
        output_files[aggregation] = out_file

    return output_files


def compile_event_file(product_files):
    os.makedirs(os.path.dirname(COMPILED_FILE), exist_ok=True)

    datasets = []
    for path in product_files:
        if path is None:
            continue
        datasets.append(xr.open_dataset(path))

    if not datasets:
        raise RuntimeError("No event products were available for compilation.")

    try:
        ds_compiled = xr.merge(datasets, compat="override")
        ds_compiled.attrs["source"] = "ERA5T"
        ds_compiled.attrs["event"] = EVENT_NAME
        ds_compiled.attrs["region"] = REGION_NAME
        ds_compiled.attrs["start_date"] = START_DATE
        ds_compiled.attrs["end_date"] = END_DATE
        ds_compiled.to_netcdf(COMPILED_FILE)
    finally:
        for ds in datasets:
            ds.close()

    return COMPILED_FILE


def run_event():
    print(Cdo().version())
    print(f"[INFO] Processing ERA5T event: {START_DATE} to {END_DATE}")
    print(f"[INFO] Region: {REGION_NAME}")

    product_files = []

    print("[INFO] Computing tasmax and tasmax_hour")
    tasmax_file = process_tasmax_event()
    product_files.append(tasmax_file)
    print(f"  -> wrote {tasmax_file}")

    print("[INFO] Computing q/t/z daily extremes, daily means, and at-tasmax fields")
    results = Parallel(
        n_jobs=N_JOBS_VARIABLES,
        verbose=10,
        backend="multiprocessing",
    )(
        delayed(process_variable_event)(config)
        for config in VARIABLE_CONFIGS
    )

    for result in results:
        product_files.extend(path for path in result.values() if path is not None)

    compiled = compile_event_file(product_files)
    print(f"[DONE] Final compiled ERA5T event file written to: {compiled}")
    print(f"[INFO] Per-product files are in: {FINAL_PATH}")
    return compiled


if __name__ == "__main__":
    run_event()